In [6]:
#Packages Used
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from sympy import *


In [4]:
# Key Constants

rho = 1025                 # seawater density [kg/m^3]
g = 9.81                   # gravity [m/s^2]


In [ ]:
# Analysing the Turbine

var("t x_s z_s phi_s phi_t")      
#phi_t tower angle (to vertical)

# Substructure Variables
x_s = Function("x_s")(t)    #Surge
phi_s = Function("phi_s")(t)    #roll
z_s = Function("z_s")(t)    #Heave

phi_t = Function("phi_t")(t)

# Constant Distances
Lx_st =  0 #Horizontal distance from substructure center to tower base
Lz_st =  89.92 + 10 #Vertical distance from substructure center to tower base
L_t = 120 # Tower Length


# Nacelle Positions
x_t = x_s + Lx_st*cos(phi_s) - Lz_st*sin(phi_s) - L_t*sin(phi_t)
z_t = z_s + Lx_st*sin(phi_s) + Lz_st*cos(phi_s) + L_t*cos(phi_t)

# Relative tower angle ->Later Energy Relations
phi_st = phi_s - phi_t

# Compute/define the velocities here
x_s_dot = diff(x_s, t)
z_s_dot = diff(z_s, t)
phi_s_dot = diff(phi_s, t)

x_t_dot = diff(x_t, t)
z_t_dot = diff(z_t, t)
phi_t_dot = diff(phi_t, t) # will not be used for point masses

phi_st_dot = diff(phi_st, t) # velocity will not be used


var("rho_s B_s H_s W_s m_t")
# substructure density, breadth, height, width;

# Turbine Mass
m_Rotor = 1.1e5           # mass of the rotor [kg]
m_Nacelle = 2.4e5         # mass of the nacelle [kg]
m_Tower = 3.47e5           # mass of the tower [kg
m_t = m_Rotor + m_Nacelle + m_Tower  # total mass of the turbine [kg]


# Spar Buoy Properties
m = 7.466e6                # total mass (Including Ballast) [kg]
R = 4.7                    # radius [m] - 
print("Check the radius used")
draft = 120                # draft [m]


# Define the kinetic energy here (T)
m_s = 7466.33e3 #Substructure mass, including ballast
# Substructure Radius
r_s = 4.7 # Assumed constant throughout
H_s = 120 + 10 # Spar Buoy Length
rho_s = m_s/(pi*r_s**2*H_s) # Substructure density, assuming uniform


J_s = Ixx = 4.229e9              # Roll inertia [kg m^2]



T_s = 1/2*m_s*(x_s_dot**2 + z_s_dot**2) + 1/2*J_s*(phi_s_dot**2)
T_t = 1/2*m_t*(x_t_dot**2 + z_t_dot**2) + 1/2*0*(phi_t_dot**2) # point mass so J_t = 0
T = T_s+T_t


### Potential energy:
var("rho_w g kr_st k_h")
# water density, rotational spring stiffness
# Define the potential energy here (V)


k_s = rho_w*g*B_s*W_s # Approximate hydrostatic restoring stiffness (linearized buoyancy)

draft_s = (B_s*W_s*H_s*rho_s)/(B_s*W_s*rho_w)
KB_s = draft_s/2 # COB, due to constant shape
nabla_s = B_s*W_s*draft_s # Submerged volume, taken in neutral position
J_sub = 1/12*W_s*B_s**3
BM_s = J_sub/nabla_s
KG_s = H_s/2 # COG, due to uniform weight
GM_s = KB_s + BM_s - KG_s
kr_s = rho_w*g*nabla_s*GM_s # Nm/rad

V_s = m_s*g*z_s + 1/2*k_h*x_s**2 + 1/2*k_s*z_s**2 + 1/2*kr_s*phi_s**2
V_t = m_t*g*z_t + 1/2*kr_st*phi_st**2 # need relative angle for this spring
V = V_s + V_t






SyntaxError: invalid syntax (3787648569.py, line 15)

In [ ]:




# Intertial Properties
Iyy = 4.229e9              # Pitch inertia [kg m^2]
Izz = 1.642e9              # Yaw inertia [kg m^2]


#~~~~~~~~~~~~~~~~~~~~~~~
# RIGID BODY MASS MATRIX
#~~~~~~~~~~~~~~~~~~~~~~~

M_RB = np.diag([
    m,
    m,
    m,
    Ixx,
    Iyy,
    Izz
])
print(M_RB)













Check the radius used
[[7.466e+06 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00]
 [0.000e+00 7.466e+06 0.000e+00 0.000e+00 0.000e+00 0.000e+00]
 [0.000e+00 0.000e+00 7.466e+06 0.000e+00 0.000e+00 0.000e+00]
 [0.000e+00 0.000e+00 0.000e+00 4.229e+09 0.000e+00 0.000e+00]
 [0.000e+00 0.000e+00 0.000e+00 0.000e+00 4.229e+09 0.000e+00]
 [0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 1.642e+09]]


In [ ]:


# Hydrostatic restoring
waterplane_area = np.pi * R**2

# Hydrostatic stiffness
K_heave = rho * g * waterplane_area

# Pitch restoring stiffness
GM = 30                    # metacentric height [m]
K_pitch = m * g * GM
